# Phase 06B.03 — Traffic-Aware Temporal Grounding Preflight

Freeze encoder/input provenance, 32 deterministic candidates, train-only support supervision, and U/QTG/TATG/RND/ORACLE arms.

**Immutable gates:** `L32-F1`; 298 frozen validation samples; train-side checkpoint selection only; no public-test access. Missing human/input artifacts produce an explicit status and stop—no synthetic labels or provenance.

## 1. Selected-track gate

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
def write_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path
from phase06a_common import sha256_file,sha256_json
from phase06b_common import load_json,validate_temporal_input_manifest
P=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; OUT=ROOT/"outputs/phase06b/traffic_temporal_grounding/preflight"
IM=ROOT/"outputs/phase06b/traffic_temporal_grounding/temporal_input_manifest.json"; OUT.mkdir(parents=True,exist_ok=True)
if not P.is_file(): raise RuntimeError("Lock Phase06B_01 first")
protocol=load_json(P); selected=protocol.get("status")=="locked" and protocol.get("selected_track")=="traffic_temporal_grounding"
if not selected: write_json(OUT/"PHASE06B_03_STATUS.json",{"status":"not_selected","selected_track":protocol.get("selected_track")})
print("selected:",selected)

## 2. Frozen encoders and train-only support gate

In [ ]:
if selected and not IM.is_file():
 write_json(OUT/"temporal_input_manifest.template.json",{"visual_encoder":"","visual_encoder_revision":"","question_encoder":"",
 "question_encoder_revision":"","candidate_count":32,"support_annotation_split":"train","support_annotations_path":"data/train/...",
 "support_annotations_sha256":"","feature_bank_schema_version":1})
 write_json(OUT/"PHASE06B_03_STATUS.json",{"status":"awaiting_temporal_input_manifest"})
 raise RuntimeError("Selected temporal track requires frozen inputs")
if selected:
 inputs=validate_temporal_input_manifest(load_json(IM)); support=ROOT/inputs["support_annotations_path"]
 if not support.is_file() or sha256_file(support)!=inputs["support_annotations_sha256"]: raise ValueError("Support file/hash mismatch")

## 3. Freeze matrix
ORACLE-k is diagnostic only and forbidden from winner/public-test inference.

In [ ]:
if selected:
 arms={}
 selectors={"U":"uniform","QTG":"question_guided","TATG":"traffic_aware_question_guided","RND":"seeded_random","ORACLE":"human_support"}
 for family in selectors:
  for k in [1,3,8]: arms[f"{family}-{k}"]={"selector":selectors[family],"traffic_features":family=="TATG","k":k,"winner_eligible":family!="ORACLE"}
 temporal={"status":"locked","selected_track":"traffic_temporal_grounding","parent_protocol_sha256":sha256_json(protocol),
 "input_manifest_sha256":sha256_file(IM),"control":"L32-F1","candidate_count":32,"arms":arms,
 "checkpoint_selection":"group-safe train/inner-dev only","grounding_metrics":["Recall@k","mAP","nDCG","temporal_distance"],
 "forbidden":["validation support for selection","validation labels for checkpoint selection","public-test access"]}
 write_json(OUT/"traffic_temporal_grounding_protocol.json",temporal)
 write_json(OUT/"PHASE06B_03_STATUS.json",{"status":"complete","protocol_sha256":sha256_json(temporal)})
 temporal